# Chapter 8 — Dimensionality Reduction
Companion notebook to `part1_fundamentals/08_dimensionality_reduction.md`.
Uses MNIST (downloads the first time) and a synthetic Swiss Roll (no download needed).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml, make_swiss_roll
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import time

## PCA on MNIST: how many dimensions to keep 95% variance?

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"][:20000], mnist["target"][:20000]   # subsample for speed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pca = PCA(n_components=0.95)
X_train_reduced = pca.fit_transform(X_train)
print("Original dims:", X_train.shape[1], "-> Reduced dims:", X_train_reduced.shape[1])
print("Explained variance kept:", pca.explained_variance_ratio_.sum())

In [ ]:
# Compare training time / accuracy: full data vs PCA-reduced data
X_test_reduced = pca.transform(X_test)

for name, Xtr, Xte in [("Full (784-d)", X_train, X_test), ("PCA (95%)", X_train_reduced, X_test_reduced)]:
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    t0 = time.time()
    clf.fit(Xtr, y_train)
    train_time = time.time() - t0
    acc = clf.score(Xte, y_test)
    print(f"{name:15s} train_time={train_time:.1f}s  test_accuracy={acc:.3f}")

## PCA compression / reconstruction

In [ ]:
pca_154 = PCA(n_components=154)
X_reduced_154 = pca_154.fit_transform(X_train)
X_recovered = pca_154.inverse_transform(X_reduced_154)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_train[0].reshape(28, 28), cmap="binary"); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(X_recovered[0].reshape(28, 28), cmap="binary"); axes[1].set_title("Reconstructed (154-d)"); axes[1].axis("off")
plt.show()

## Unrolling a nonlinear manifold: PCA vs Kernel PCA vs LLE

In [ ]:
X_swiss, t_swiss = make_swiss_roll(n_samples=1500, noise=0.05, random_state=42)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=t_swiss, cmap="jet")
ax.set_title("Original Swiss Roll (3D)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pca_2d = PCA(n_components=2).fit_transform(X_swiss)
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=t_swiss, cmap="jet"); axes[0].set_title("Plain PCA (flattens, tangles layers)")

kpca_2d = KernelPCA(n_components=2, kernel="rbf", gamma=0.04).fit_transform(X_swiss)
axes[1].scatter(kpca_2d[:, 0], kpca_2d[:, 1], c=t_swiss, cmap="jet"); axes[1].set_title("Kernel PCA (rbf)")

lle_2d = LocallyLinearEmbedding(n_components=2, n_neighbors=10).fit_transform(X_swiss)
axes[2].scatter(lle_2d[:, 0], lle_2d[:, 1], c=t_swiss, cmap="jet"); axes[2].set_title("LLE (unrolls cleanly)")

plt.show()

## ✍️ Practice
1. Above: compare training time/accuracy full vs PCA-reduced — was the trade-off worth it here?
2. Explain why maximizing variance = minimizing reconstruction error (Pythagorean theorem intuition).
3. Above: which method unrolled the Swiss Roll most cleanly — plain PCA, Kernel PCA, or LLE? Why?
4. Try KernelPCA with a few different gamma values on the Swiss Roll — how sensitive is the result?

In [ ]:
# Your code / notes here